# Freeman two-component decomposition on geocoded data

This notebook checks the visual appearance and NaN behavior of the ground and volume outputs in SAR and geographic coordinates. Numerical comparison with C uses the separate SLC parity notebook.

Geocoding resamples the coherent matrix data, so these images are a visual robustness check, not evidence of numerical or physical equivalence with the SLC decomposition.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from dask.diagnostics import ProgressBar

from polsarpro.decompositions import freeman2
from polsarpro.io import open_netcdf_beam

# Change to your data paths.
input_alos_slc = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_alos_geo = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_geocoded_T3_7_look_az.nc")

# Validated on the ALOS-1 SLC test data; adjust for other datasets if needed.
# condition_threshold = 0.01
condition_threshold = None

In [ ]:
slc = open_netcdf_beam(input_alos_slc)
geo = open_netcdf_beam(input_alos_geo)

## Apply the decomposition

Use the same averaging windows as the Freeman three-component geocoded test. The geocoded input has already been multilooked in azimuth, so it receives less additional filtering. The optional threshold marks poorly conditioned estimates as NaN; set it to `None` to disable this exclusion.

In [ ]:
# Persist retains Dask arrays for repeated plotting and mask checks.
with ProgressBar():
    res_slc = freeman2(
        slc, boxcar_size=(7, 7), condition_threshold=condition_threshold
    ).persist()
with ProgressBar():
    res_geo = freeman2(
        geo, boxcar_size=(3, 3), condition_threshold=condition_threshold
    ).persist()

## Display outputs

The SLC display is subsampled in azimuth for readability. The two panels are not aligned pixel by pixel. Check for unexpected artifacts, especially near masked regions.

In [ ]:
for var in res_geo.data_vars:
    fig, axes = plt.subplots(1, 2, figsize=(10, 10))
    fig.suptitle(var)
    for ax, result, title in zip(
        axes, (res_slc[var][::8], res_geo[var]), ("SLC", "Geocoded")
    ):
        image = ax.imshow(result, interpolation="none", vmin=0, vmax=1)
        ax.set_title(title)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.show()

## Check NaN behavior

Invalid input pixels should remain NaN in both outputs. Additional NaNs may arise from the conditioning threshold or invalid averaged matrix values. Display the geocoded output mask alongside the input mask to inspect their spatial distribution.

In [ ]:
with ProgressBar():
    geo_valid = np.isfinite(geo.to_array()).all("variable").compute()
    geo_finite = np.isfinite(res_geo.to_array()).all("variable").compute()
    geo_nan = res_geo.to_array().isnull().all("variable").compute()

assert bool(geo_nan.where(~geo_valid, True).all())
assert bool((res_geo.ground.isnull() == res_geo.volume.isnull()).all().compute())

{
    "invalid_input": int((~geo_valid).sum()),
    "retained_output": int(geo_finite.sum()),
    "additional_exclusions": int((geo_valid & ~geo_finite).sum()),
}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, mask, title in zip(
    axes, (~geo_valid, geo_nan), ("Invalid geocoded input", "NaN decomposition output")
):
    image = ax.imshow(mask, interpolation="none", vmin=0, vmax=1, cmap="gray_r")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.show()